# CNN with Featured data Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)


In [ ]:
# Import necessary libraries
import os
import json
import numpy as np
import pandas as pd
import h5py
import tensorflow as tf
from tensorflow.keras import layers, models
import keras.backend as K
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from dataclasses import dataclass
from typing import List, Tuple, Dict
from sklearn.model_selection import KFold, GroupKFold




print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TF: 2.10.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [6]:
# Reprodutibilidade
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [28]:

# chunking (5 min com overlap)
CHUNK_SEC  = 300
STRIDE_SEC = 60   # overlap = 240s (80%)

FS = 1  # 1Hz para features e labels

# treino
N_SPLITS   = 5
BATCH_SIZE = 16
EPOCHS     = 50
LR         = 1e-3



## Model Choice

As a baseline model, we implement the reference architecture provided in the challenge description:
a **3-layer 1D Convolutional Neural Network (CNN)**.

The architecture consists of:
- Two convolution + max-pooling blocks (each pooling with factor 10),
- One final convolution layer producing a 1 Hz probability sequence (90 values per window).

This baseline is appropriate because:
1. It is the official benchmark model of the challenge.
2. It processes raw PSG signals sampled at 100 Hz without handcrafted features.
3. It outputs a 1 Hz segmentation mask aligned with the provided ground-truth labels.
4. It provides a simple reference point for comparison with more complex architectures.

The model does not incorporate recurrence or attention mechanisms by design.




## Feature Selection

Each training example consists of:
- **8 PSG channels** sampled at **100 Hz**, resulting in input tensors of shape `(9000, 8)` for each 90-second window.
- A **binary segmentation mask** of shape `(90,)`, sampled at 1 Hz.

No handcrafted features are used.
The only preprocessing step applied is **signal normalization**, performed prior to training.

Subject identifiers are provided separately and are used exclusively to perform a
**subject-wise train/validation split**, preventing data leakage.



In [4]:
# ========================================
# 1. CARREGAR DADOS (CORRIGIR y_raw → y)
# ========================================
IN_PATH = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\02_Feature_Eng\nights_train_features_1hz.h5"

with h5py.File(IN_PATH, "r") as f:
    X_feat = f["X_feat"][:]         # (22, 18000, n_feat)
    y = f["y_nights"][:]            # (22, 18000) @ 1Hz 
    subj = f["subject_ids"][:]
    feature_names = [x.decode() for x in f["feature_names"][:]]

print("X_feat:", X_feat.shape)
print("y:", y.shape)  
print("n_features:", X_feat.shape[2])
print("subj:", subj.shape)

X_feat: (22, 18000, 18)
y: (22, 18000)
n_features: 18
subj: (22,)


In [9]:
print("X_feat:", X_feat.shape, X_feat.dtype)
print("y     :", y.shape, y.dtype)
print("y min/max/mean:", y.min(), y.max(), y.mean())

assert X_feat.shape[:2] == y.shape
assert X_feat.shape[2] == 18
assert y.min() >= 0 and y.max() <= 1


X_feat: (22, 18000, 18) float32
y     : (22, 18000) int8
y min/max/mean: 0 1 0.06872222222222223


In [10]:
def binary_to_events(y_1hz: np.ndarray, min_len: int = 1) -> List[Tuple[int,int]]:
    yb = (y_1hz > 0.5).astype(np.uint8)
    events = []
    n = len(yb)
    i = 0
    while i < n:
        if yb[i] == 1:
            j = i
            while j < n and yb[j] == 1:
                j += 1
            if (j - i) >= min_len:
                events.append((i, j))
            i = j
        else:
            i += 1
    return events

gt_events_by_night = {nid: binary_to_events(y[nid], min_len=1) for nid in range(y.shape[0])}

for nid in range(3):
    print(f"night {nid}: n_events={len(gt_events_by_night[nid])} pos%={y[nid].mean()*100:.2f}%")


night 0: n_events=12 pos%=0.99%
night 1: n_events=16 pos%=2.90%
night 2: n_events=96 pos%=9.56%


In [11]:
def chunk_generator_with_meta(X_nights, y_nights, chunk_sec=300, stride_sec=60, fs=1):
    chunk_len  = chunk_sec * fs
    stride_len = stride_sec * fs

    n_nights, T, C = X_nights.shape
    assert y_nights.shape[0] == n_nights
    assert y_nights.shape[1] == T

    for night_id in range(n_nights):
        X = X_nights[night_id]    # (T, C) 1Hz
        yy = y_nights[night_id]   # (T,) 1Hz

        for start in range(0, T - chunk_len + 1, stride_len):
            end = start + chunk_len
            X_chunk = X[start:end]      # (300, C)
            y_chunk = yy[start:end]     # (300,)
            if y_chunk.shape[0] != chunk_sec:
                continue
            yield (
                X_chunk.astype("float32"),
                y_chunk.astype("float32"),
                np.int32(night_id),
                np.int32(start)  # start_sec
            )


In [12]:
def make_tf_dataset_with_meta(X_nights, y_nights, chunk_sec=300, stride_sec=60, fs=1,
                              batch_size=16, shuffle_buffer=4096, training=True):

    output_signature = (
        tf.TensorSpec(shape=(chunk_sec, X_nights.shape[2]), dtype=tf.float32),
        tf.TensorSpec(shape=(chunk_sec,), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )

    ds = tf.data.Dataset.from_generator(
        lambda: chunk_generator_with_meta(X_nights, y_nights, chunk_sec, stride_sec, fs),
        output_signature=output_signature
    )

    if training:
        ds = ds.shuffle(shuffle_buffer, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# quick test
ds_test = make_tf_dataset_with_meta(X_feat[:2], y[:2], CHUNK_SEC, STRIDE_SEC, FS, batch_size=2, training=False)
for Xb, yb, nb, sb in ds_test.take(1):
    print("X:", Xb.shape, "y:", yb.shape, "night_id:", nb.numpy(), "start:", sb.numpy())


X: (2, 300, 18) y: (2, 300) night_id: [0 0] start: [ 0 60]


## Implementation

Below we implement the 3-layer CNN baseline as specified by the challenge.
Two Conv1D + MaxPool1D blocks downsample the 100 Hz signal to 1 Hz, and the final
Conv1D layer outputs class probabilities for each of the 90 seconds.


CNN Baseline model

In [13]:
def build_baseline_cnn(n_samples, n_channels):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(n_samples, n_channels)),
        tf.keras.layers.Conv1D(32, 5, padding='same', activation='relu'),
        tf.keras.layers.Conv1D(32, 5, padding='same', activation='relu'),
        tf.keras.layers.Conv1D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.Conv1D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.Conv1D(1, 1, padding='same', activation='sigmoid'),
        tf.keras.layers.Lambda(lambda t: tf.squeeze(t, axis=-1))
    ])
    return model

model = build_baseline_cnn(CHUNK_SEC, X_feat.shape[2])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d (Conv1D)             (None, 300, 32)           2912      
                                                                 
 conv1d_1 (Conv1D)           (None, 300, 32)           5152      
                                                                 
 conv1d_2 (Conv1D)           (None, 300, 64)           6208      
                                                                 
 conv1d_3 (Conv1D)           (None, 300, 64)           12352     
                                                                 
 conv1d_4 (Conv1D)           (None, 300, 1)            65        
                                                                 
 lambda (Lambda)             (None, 300)               0         
                                                                 
Total params: 26,689
Trainable params: 26,689
Non-traina

Loss Function (Weighted BCE)

In [14]:
positive_ratio = float(np.mean(y))
neg_ratio = 1.0 - positive_ratio
pos_weight = neg_ratio / (positive_ratio + 1e-12)
print("Positive ratio:", positive_ratio, "| pos_weight:", pos_weight)

def weighted_bce(pos_weight):
    def loss(y_true, y_pred):
        bce = K.binary_crossentropy(y_true, y_pred)
        w = 1.0 + (pos_weight - 1.0) * y_true
        return K.mean(bce * w)
    return loss




Positive ratio: 0.06872222222222223 | pos_weight: 13.551333872074434


In [15]:
def predict_chunks_with_meta(model, ds):
    preds, y_true, night_ids, start_secs = [], [], [], []
    for Xb, yb, nb, sb in ds:
        pb = model.predict(Xb, verbose=0)  # (B, 300)
        preds.append(pb.astype(np.float32))
        y_true.append(yb.numpy().astype(np.float32))
        night_ids.append(nb.numpy().astype(np.int32))
        start_secs.append(sb.numpy().astype(np.int32))

    return (np.concatenate(preds, axis=0),
            np.concatenate(y_true, axis=0),
            np.concatenate(night_ids, axis=0),
            np.concatenate(start_secs, axis=0))

def stitch_night_probs(chunks_probs, chunks_starts, night_len_sec, chunk_sec=300):
    acc = np.zeros(night_len_sec, dtype=np.float32)
    cnt = np.zeros(night_len_sec, dtype=np.float32)
    for p, s in zip(chunks_probs, chunks_starts):
        e = min(s + chunk_sec, night_len_sec)
        L = e - s
        if L <= 0:
            continue
        acc[s:e] += p[:L]
        cnt[s:e] += 1.0
    return acc / np.maximum(cnt, 1e-6)


In [16]:
@dataclass
class PostParams:
    t: float = 0.58
    min_len: int = 10
    gap_fill: int = 6
    smooth_w: int = 3

class PostProcessor:
    def __init__(self, p: PostParams):
        self.p = p

    def __call__(self, score: np.ndarray) -> List[Tuple[int,int]]:
        x = score.astype(np.float32)

        # smoothing
        if self.p.smooth_w > 1:
            w = int(self.p.smooth_w)
            k = np.ones(w, dtype=np.float32) / w
            x = np.convolve(x, k, mode="same")

        b = (x >= self.p.t).astype(np.uint8)

        # gap fill
        gf = int(self.p.gap_fill)
        if gf > 0:
            n = len(b)
            i = 0
            while i < n:
                if b[i] == 1:
                    j = i
                    while j < n and b[j] == 1: j += 1
                    k = j
                    while k < n and b[k] == 0: k += 1
                    if k < n and (k - j) <= gf:
                        b[j:k] = 1
                    i = k
                else:
                    i += 1

        # extract events
        events = []
        ml = int(self.p.min_len)
        n = len(b)
        i = 0
        while i < n:
            if b[i] == 1:
                j = i
                while j < n and b[j] == 1: j += 1
                if (j - i) >= ml:
                    events.append((i, j))
                i = j
            else:
                i += 1
        return events

def match_events(gt: List[Tuple[int,int]], pr: List[Tuple[int,int]], min_overlap_sec: int = 1):
    gt_used = np.zeros(len(gt), dtype=bool)
    tp = 0
    for (ps, pe) in pr:
        for i, (gs, ge) in enumerate(gt):
            if gt_used[i]:
                continue
            overlap = max(0, min(pe, ge) - max(ps, gs))
            if overlap >= min_overlap_sec:
                tp += 1
                gt_used[i] = True
                break
    fp = len(pr) - tp
    fn = len(gt) - tp
    return tp, fp, fn


In [ ]:
import os
import numpy as np
from sklearn.model_selection import KFold
from itertools import product

# --- helpers do pós (se você já tem, pode manter os seus) ---
def prob_to_events(p, t=0.58, min_len=10, gap_fill=6):
    x = (p >= t).astype(np.int32)

    # gap fill
    if gap_fill > 0:
        idx1 = np.where(x == 1)[0]
        if len(idx1) > 0:
            splits = np.where(np.diff(idx1) > 1)[0]
            seg_starts = np.r_[idx1[0], idx1[splits + 1]]
            seg_ends   = np.r_[idx1[splits], idx1[-1]]
            for a_end, b_start in zip(seg_ends[:-1], seg_starts[1:]):
                gap = b_start - a_end - 1
                if 0 < gap <= gap_fill:
                    x[a_end+1:b_start] = 1

    idx = np.where(x == 1)[0]
    if len(idx) == 0:
        return []

    splits = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[splits + 1]]
    ends   = np.r_[idx[splits], idx[-1]] + 1

    events = [(int(s), int(e)) for s, e in zip(starts, ends) if (e - s) >= min_len]
    return events

def f1_from_counts(tp, fp, fn, eps=1e-9):
    prec = tp / (tp + fp + eps)
    rec  = tp / (tp + fn + eps)
    f1 = 2 * prec * rec / (prec + rec + eps)
    return f1, prec, rec

def stitch_probs(chunks_probs, chunks_starts, night_len_sec, chunk_sec):
    acc = np.zeros(night_len_sec, dtype=np.float32)
    cnt = np.zeros(night_len_sec, dtype=np.float32)
    for p, s in zip(chunks_probs, chunks_starts):
        s = int(s)
        e = min(s + chunk_sec, night_len_sec)
        L = e - s
        if L <= 0:
            continue
        acc[s:e] += p[:L]
        cnt[s:e] += 1.0
    timeline = acc / np.maximum(cnt, 1e-6)
    return timeline, cnt

def grid_search_postproc(timelines_by_night, gt_events_by_night, nights,
                         t_list, min_len_list, gap_fill_list,
                         match_events_fn, min_overlap_sec=1):
    best = None
    rows = []

    for t, min_len, gap_fill in product(t_list, min_len_list, gap_fill_list):
        TP = FP = FN = 0
        for nid in nights:
            p = timelines_by_night[nid]
            pred_events = prob_to_events(p, t=t, min_len=min_len, gap_fill=gap_fill)
            gt_events = gt_events_by_night[nid]

            # usa sua função match_events existente (mantive sua assinatura)
            tp, fp, fn = match_events_fn(gt_events, pred_events, min_overlap_sec=min_overlap_sec)
            TP += tp; FP += fp; FN += fn

        f1, prec, rec = f1_from_counts(TP, FP, FN)
        row = {"t": float(t), "min_len": int(min_len), "gap_fill": int(gap_fill),
               "TP": int(TP), "FP": int(FP), "FN": int(FN),
               "F1": float(f1), "Prec": float(prec), "Rec": float(rec)}
        rows.append(row)

        if (best is None) or (row["F1"] > best["F1"]):
            best = row

    return best, rows


def run_kfold_oof(
    X_nights, y_nights, gt_events_by_night,
    n_splits=5, batch_size=16, epochs=50, lr=1e-3,
    post_params=PostParams(), min_overlap_sec=1,
    # grid config (pode ajustar)
    grid_t_list=None, grid_min_len_list=None, grid_gap_fill_list=None,
    ckpt_dir=None,
):
    """
    Treina KFold e no fim de cada fold roda grid search de pós-processamento
    usando as nights de validação do fold (OOF daquele fold).
    Retorna um dict com tudo para EDA.
    """

    n_nights = X_nights.shape[0]
    night_len = X_nights.shape[1]  # 18000

    if ckpt_dir is None:
        ckpt_dir = os.getcwd()
    os.makedirs(ckpt_dir, exist_ok=True)

    # grid defaults (rápido -> depois você refina)
    if grid_t_list is None:
        grid_t_list = np.round(np.arange(0.30, 0.81, 0.05), 2)
    if grid_min_len_list is None:
        grid_min_len_list = [6, 8, 10, 12, 15]
    if grid_gap_fill_list is None:
        grid_gap_fill_list = [0, 2, 3, 4, 6, 8]

    results = {
        "models": [],
        "histories": [],
        "fold_nights": [],        # lista por fold
        "fold_metrics": [],       # dict por fold
        "oof": {
            "timeline": {},       # nid -> timeline
            "cnt": {},            # nid -> cnt
        }
    }

    # OOF completo
    oof_stitched = {nid: None for nid in range(n_nights)}
    oof_cnt = {nid: None for nid in range(n_nights)}

    fold_stats = []
    best_post_by_fold = {}     # fold -> best params
    grid_by_fold = {}          # fold -> grid rows

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for fold, (tr_nights, va_nights) in enumerate(kf.split(np.arange(n_nights)), 1):
        print(f"\n===== FOLD {fold}/{n_splits} =====")

        X_tr, y_tr = X_nights[tr_nights], y_nights[tr_nights]
        X_va, y_va = X_nights[va_nights], y_nights[va_nights]

        ds_tr = make_tf_dataset_with_meta(
            X_tr, y_tr, CHUNK_SEC, STRIDE_SEC, FS,
            batch_size=batch_size, training=True
        )
        ds_va = make_tf_dataset_with_meta(
            X_va, y_va, CHUNK_SEC, STRIDE_SEC, FS,
            batch_size=batch_size, training=False
        )

        model = build_baseline_cnn(CHUNK_SEC, X_nights.shape[2])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(lr),
            loss=weighted_bce(pos_weight),
            metrics=[tf.keras.metrics.AUC(curve="PR", name="auprc")]
        )

        ckpt_best = os.path.join(ckpt_dir, f"{RUN_ID}_fold{fold:02d}_best.keras")

        cbs = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
            ),
            tf.keras.callbacks.ModelCheckpoint(
                filepath=ckpt_best, save_best_only=True, monitor="val_loss", verbose=1
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
            )
        ]

        history = model.fit(
            ds_tr.map(lambda x, y, n, s: (x, y)),
            validation_data=ds_va.map(lambda x, y, n, s: (x, y)),
            epochs=epochs,
            callbacks=cbs,
            verbose=2
        )

        # registradores
        results["models"].append(model)
        results["histories"].append(history.history)
        results["fold_nights"].append(list(map(int, va_nights)))

        # --- predict chunks + meta (ESSA linha é a correta; não duplique predict) ---
        p_chunks, _, night_local, start_secs = predict_chunks_with_meta(model, ds_va)
        p_chunks = np.asarray(p_chunks).ravel()
        start_secs = np.asarray(start_secs).astype(np.int32)
        night_local = np.asarray(night_local).astype(np.int32)

        # map local night_id (0..len(X_va)-1) -> global night_id
        local_to_global = {i: int(va_nights[i]) for i in range(len(va_nights))}
        night_global = np.array([local_to_global[int(n)] for n in night_local], dtype=np.int32)

        fold_tp = fold_fp = fold_fn = 0

        # para o grid do fold
        timelines_fold = {}

        for nid in map(int, va_nights):
            idx = np.where(night_global == nid)[0]

            timeline, cnt = stitch_probs(
                p_chunks[idx], start_secs[idx],
                night_len_sec=night_len, chunk_sec=CHUNK_SEC
            )

            # salva OOF
            oof_stitched[nid] = timeline
            oof_cnt[nid] = cnt
            results["oof"]["timeline"][nid] = timeline
            results["oof"]["cnt"][nid] = cnt

            # métricas com params atuais
            post = PostProcessor(post_params)
            pred_events = post(timeline)
            gt_events = gt_events_by_night[nid]
            tp, fp, fn = match_events(gt_events, pred_events, min_overlap_sec=min_overlap_sec)

            fold_tp += tp; fold_fp += fp; fold_fn += fn

            print(
                f"night {nid:02d}: TP={tp:3d} | FP={fp:3d} | FN={fn:3d} | "
                f"true%={y_nights[nid].mean()*100:5.2f}% | pred%={(timeline>=post_params.t).mean()*100:5.2f}%"
            )

            timelines_fold[nid] = timeline

        print(f"FOLD {fold} TOTAL -> TP={fold_tp} FP={fold_fp} FN={fold_fn}")
        fold_stats.append((fold_tp, fold_fp, fold_fn))

        # --- GRID SEARCH de pós-processamento (no fold) ---
        best_post, grid_rows = grid_search_postproc(
            timelines_by_night=timelines_fold,
            gt_events_by_night=gt_events_by_night,
            nights=list(map(int, va_nights)),
            t_list=grid_t_list,
            min_len_list=grid_min_len_list,
            gap_fill_list=grid_gap_fill_list,
            match_events_fn=match_events,
            min_overlap_sec=min_overlap_sec
        )
        best_post_by_fold[fold] = best_post
        grid_by_fold[fold] = grid_rows

        print(f"[FOLD {fold}] BEST POST: {best_post}")

        # salva métricas do fold
        results["fold_metrics"].append({
            "fold": fold,
            "TP": int(fold_tp),
            "FP": int(fold_fp),
            "FN": int(fold_fn),
            "best_post": best_post
        })

    # OOF total
    TP = sum(t for t, _, _ in fold_stats)
    FP = sum(f for _, f, _ in fold_stats)
    FN = sum(n for _, _, n in fold_stats)

    print("\n===== OOF TOTAL =====")
    print(f"TP={TP} FP={FP} FN={FN}")

    out = {
        "results": results,
        "fold_stats": fold_stats,
        "oof_stitched": oof_stitched,
        "oof_cnt": oof_cnt,
        "best_post_by_fold": best_post_by_fold,
        "grid_by_fold": grid_by_fold,
        "oof_total": {"TP": int(TP), "FP": int(FP), "FN": int(FN)},
    }
    return out



In [27]:
post_params = PostParams(t=0.48, min_len=10, gap_fill=3, smooth_w=3)

fold_stats, oof_stitched, results = run_kfold_oof(
    X_feat, y, gt_events_by_night,
    n_splits=N_SPLITS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    lr=LR,
    post_params=post_params,
    min_overlap_sec=1
)



===== FOLD 1/5 =====
Epoch 1/20

Epoch 1: val_loss improved from inf to 7.88403, saving model to C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\3_Model\3lcnn_featured\20260121_133440_best.keras
315/315 - 7s - loss: 10.4102 - auprc: 0.0805 - val_loss: 7.8840 - val_auprc: 0.0433 - lr: 0.0010 - 7s/epoch - 23ms/step
Epoch 2/20

Epoch 2: val_loss improved from 7.88403 to 2.82379, saving model to C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\3_Model\3lcnn_featured\20260121_133440_best.keras
315/315 - 8s - loss: 5.6555 - auprc: 0.0793 - val_loss: 2.8238 - val_auprc: 0.0480 - lr: 0.0010 - 8s/epoch - 24ms/step
Epoch 3/20

Epoch 3: val_loss improved from 2.82379 to 1.10367, saving model to C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\3_Model\3lcnn_featured\20260121_133440_best.keras
315/315 - 8s - loss: 2.1902 - auprc: 0.0856 - val_loss: 1.1037 - val_auprc: 0.0467 - lr: 0.0010 - 8s/epoch - 25ms/step
Epoch 4/20

Epoch 4: val_loss improved from 1.10367 to 1.07670, saving model to C:\User

In [29]:
def stitch_with_counts(chunks_probs, chunks_starts, night_len_sec, chunk_sec=300):
    acc = np.zeros(night_len_sec, dtype=np.float32)
    cnt = np.zeros(night_len_sec, dtype=np.float32)
    for p, s in zip(chunks_probs, chunks_starts):
        e = min(s + chunk_sec, night_len_sec)
        L = e - s
        if L <= 0:
            continue
        acc[s:e] += p[:L]
        cnt[s:e] += 1.0
    timeline = acc / np.maximum(cnt, 1e-6)
    return timeline, cnt

# dentro do loop do fold, por night:
timeline, cnt = stitch_with_counts(p_chunks[idx], start_secs[idx], night_len, CHUNK_SEC)
print(nid, "cnt min/mean/max:", cnt.min(), cnt.mean(), cnt.max())


NameError: name 'p_chunks' is not defined

In [31]:
import numpy as np
from sklearn.model_selection import KFold

n_nights = X_feat.shape[0]  # 22
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

fold_nights = {}
for fold, (_, va_idx) in enumerate(kf.split(np.arange(n_nights)), 1):
    fold_nights[fold] = list(map(int, va_idx))

fold_nights


{1: [0, 1, 8, 13, 15],
 2: [3, 4, 5, 11, 20],
 3: [12, 16, 17, 18],
 4: [2, 7, 9, 21],
 5: [6, 10, 14, 19]}

In [32]:
import pandas as pd
import numpy as np

t = 0.58

rows = []
for fold, nights in fold_nights.items():  # {1:[...],2:[...],...}
    for nid in nights:
        tl = oof_stitched.get(nid)  # timeline OOF (1Hz)
        if tl is None:
            continue
        rows.append({
            "fold": fold,
            "night": nid,
            "true_pct": y[nid].mean()*100,
            "pred_pct": (tl >= t).mean()*100,
            "p50": np.percentile(tl, 50),
            "p95": np.percentile(tl, 95),
            "p99": np.percentile(tl, 99),
            "max": float(np.max(tl)),
        })

df = pd.DataFrame(rows).sort_values(["fold","night"])
df


,fold,night,true_pct,pred_pct,p50,p95,p99,max
0,1,0,0.988889,2.011111,0.464984,0.551308,0.601270,0.999816
1,1,1,2.900000,3.494444,0.481886,0.560919,0.663984,1.000000
2,1,8,13.772222,3.416667,0.485519,0.562907,0.640336,0.999994
3,1,13,1.633333,1.150000,0.437951,0.531934,0.590474,0.881917
4,1,15,3.405556,2.477778,0.432903,0.557399,0.624714,1.000000
5,2,3,7.966667,0.933333,0.479201,0.479387,0.567353,0.992882
6,2,4,27.238889,0.366667,0.479201,0.479387,0.524395,1.000000
7,2,5,7.966667,0.016667,0.479201,0.479201,0.487493,0.829247
8,2,11,2.116667,1.650000,0.479201,0.479387,0.728405,0.998865
9,2,20,1.388889,0.438889,0.479201,0.479201,0.487786,0.921669


In [33]:
df.groupby("fold")[["true_pct","pred_pct","p50","p95","p99","max"]].agg(["mean","std","min","max"])


true_pct                                  pred_pct                      \
          mean        std       min        max      mean       std       min   
fold                                                                           
1     4.540000   5.250466  0.988889  13.772222  2.510000  0.986243  1.150000   
2     9.335556  10.482592  1.388889  27.238889  0.681111  0.632731  0.016667   
3     9.518056   7.976748  0.744444  19.211111  0.648611  0.402931  0.150000   
4     6.226389   3.592678  1.127778   9.561111  9.148611  4.736309  4.855556   
5     4.708333   4.264799  1.150000  10.305556  4.913889  2.705035  2.900000   

                      p50            ...       p95                 p99  \
            max      mean       std  ...       min       max      mean   
fold                                 ...                                 
1      3.494444  0.460649  0.024359  ...  0.531934  0.562907  0.624156   
2      1.650000  0.479201  0.000000  ...  0.479201  0.479387  0.559086   
3      1.116667  0.481738  0.000000  ...  0.481738  0.483111  0.540377   
4     15.550000  0.483877  0.000000  ...  0.576601  0.708899  0.811322   
5      8.805556  0.474982  0.000000  ...  0.488451  0.641611  0.740444   

                                         max                                
           std       min       max      mean       std       min       max  
fold                                                                        
1     0.029608  0.590474  0.663984  0.976345  0.052787  0.881917  1.000000  
2     0.100187  0.487493  0.728405  0.948532  0.074330  0.829247  1.000000  
3     0.039281  0.502791  0.595420  0.938406  0.081137  0.828261  1.000000  
4     0.077295  0.735546  0.901445  0.991477  0.005380  0.985397  0.998464  
5     0.062452  0.689374  0.823538  0.979603  0.016224  0.959292  0.994126  

[5 rows x 24 columns]

In [34]:
print("p_chunks stats:", p_chunks.min(), p_chunks.mean(), p_chunks.max())
print("p_chunks p50/p95:", np.percentile(p_chunks, 50), np.percentile(p_chunks, 95))


NameError: name 'p_chunks' is not defined

Compile & Train

In [20]:

# ✅ Definir diretório uma única vez
OUT_DIR = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\3_Model\3lcnn_featured"
os.makedirs(OUT_DIR, exist_ok=True)

# ✅ Usar timestamp para diferenciar runs
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
ckpt_best = os.path.join(OUT_DIR, f"{run_id}_best.keras")
ckpt_last = os.path.join(OUT_DIR, f"{run_id}_last.keras")

print(f"✅ Checkpoints serão salvos em:\n   {OUT_DIR}")
print(f"   Best: {ckpt_best}")
print(f"   Last: {ckpt_last}")

# ✅ Callbacks corrigidos
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", 
        patience=10, 
        restore_best_weights=True, 
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=ckpt_best,              # ✅ Usa variável correta
        save_best_only=True, 
        monitor="val_loss", 
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", 
        factor=0.5, 
        patience=3, 
        min_lr=1e-6, 
        verbose=1
    )
]

✅ Checkpoints serão salvos em:
   C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\3_Model\3lcnn_featured
   Best: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\3_Model\3lcnn_featured\20260121_133440_best.keras
   Last: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\3_Model\3lcnn_featured\20260121_133440_last.keras


In [ ]:
#save history as pickle
import pickle
with open(os.path.join(OUT_DIR, f"{run_id}_history.pkl"), "wb") as f:
    pickle.dump(history.history, f)



In [ ]:
plt.plot(history3.history["loss"], label="train")
plt.plot(history3.history["val_loss"], label="val")
plt.legend()
plt.show()


## Evaluation

Model performance is evaluated using an **event-based F1-score**, consistent with the
official metric of the Dreem Sleep Apnea Challenge.

Predictions are first reconstructed at **1 Hz over the full night** by aggregating
overlapping window-level outputs using an overlap-and-average strategy. The resulting
probability sequence is then thresholded to obtain a binary apnea mask.

Apnea events are extracted as contiguous segments in the binary mask and compared to the
ground-truth annotations using an **Intersection over Union (IoU)** criterion. A predicted
event is considered a true positive if its IoU with a reference event is greater than or
equal to **0.3**.

The final score is computed as an **event-level F1-score**, aggregating true positives,
false positives, and false negatives across all validation nights.

For completeness, window-level metrics such as accuracy and recall may be reported for
debugging purposes, but **they are not used for model selection**, as they do not reflect
the temporal structure of apnea events.



In [ ]:
best_frozen = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\2_BaselineModel\3lcnn_night\20251214_014656_BEST_frozen.keras"

model = tf.keras.models.load_model(best_frozen, compile = False)
print("Modelo carregado:", best_frozen)


In [ ]:
def predict_nights_overlap_mean(
    model,
    X_nights,                 # (n_nights, 1800000, 8)
    chunk_sec=300,
    stride_sec=60,
    fs=1,
    batch_size=4
):
    n_nights, T, C = X_nights.shape
    chunk_len = chunk_sec * fs        # 300
    stride_len = stride_sec * fs      # 60
    T1 = T // fs                      # 18000

    y_sum = np.zeros((n_nights, T1), dtype=np.float32)
    y_cnt = np.zeros((n_nights, T1), dtype=np.float32)

    for n in range(n_nights):
        X = X_nights[n]  # (1800000, 8)
        starts = range(0, T - chunk_len + 1, stride_len)

        batch_X = []
        batch_ranges = []

        for start in starts:
            end = start + chunk_len
            x_chunk = X[start:end]          # (30000, 8)

            s1 = start // fs
            e1 = end // fs                  # 300 pontos

            batch_X.append(x_chunk)
            batch_ranges.append((s1, e1))

            if len(batch_X) == batch_size:
                preds = model.predict(np.stack(batch_X).astype("float32"), verbose=0)  # (B, 300)
                for p, (s1_, e1_) in zip(preds, batch_ranges):
                    y_sum[n, s1_:e1_] += p
                    y_cnt[n, s1_:e1_] += 1.0
                batch_X, batch_ranges = [], []

        if batch_X:
            preds = model.predict(np.stack(batch_X).astype("float32"), verbose=0)
            for p, (s1_, e1_) in zip(preds, batch_ranges):
                y_sum[n, s1_:e1_] += p
                y_cnt[n, s1_:e1_] += 1.0

    return y_sum / np.maximum(y_cnt, 1.0)


In [ ]:
y_pred_val_nights = predict_nights_overlap_mean(
    model,
    X_val,                 # (n_val, 180, 8)
    chunk_sec=300,
    stride_sec=60,
    fs=1,
    batch_size=4
)

print(y_pred_val_nights.shape)  # esperado: (n_val, 180)


In [ ]:

print("y_pred_val_nights:", y_pred_val_nights.shape)  # (n_val, 18000)
print("min/max/mean:", y_pred_val_nights.min(), y_pred_val_nights.max(), y_pred_val_nights.mean())


In [ ]:
def extract_events_from_binary_mask(binary_mask, fs=1):
    binary_mask = np.asarray(binary_mask).astype(int)
    padded = np.concatenate([[0], binary_mask, [0]])
    diff = np.diff(padded)
    starts = np.where(diff == 1)[0] / fs
    ends   = np.where(diff == -1)[0] / fs
    return [(float(s), float(e)) for s, e in zip(starts, ends)]

def jaccard_overlap(pred_events, true_events):
    # rows=true, cols=pred
    A = len(pred_events)
    B = len(true_events)
    if A == 0 or B == 0:
        return np.zeros((B, A), dtype=np.float32)

    p_start = np.array([s for s, e in pred_events])[None, :]
    p_end   = np.array([e for s, e in pred_events])[None, :]
    t_start = np.array([s for s, e in true_events])[:, None]
    t_end   = np.array([e for s, e in true_events])[:, None]

    inter = np.maximum(np.minimum(p_end, t_end) - np.maximum(p_start, t_start), 0.0)
    union = (p_end - p_start) + (t_end - t_start) - inter + 1e-12
    return (inter / union).astype(np.float32)

def tp_fp_fn(pred_events, true_events, min_iou=0.3):
    if len(pred_events) == 0:
        return 0, 0, len(true_events)
    if len(true_events) == 0:
        return 0, len(pred_events), 0

    iou = jaccard_overlap(pred_events, true_events)  # (n_true, n_pred)
    matched_true = int(np.any(iou >= min_iou, axis=1).sum())
    matched_pred = int(np.any(iou >= min_iou, axis=0).sum())

    tp = int(min(matched_true, matched_pred))
    fp = len(pred_events) - tp
    fn = len(true_events) - tp
    return tp, fp, fn

def event_f1_nights(y_pred_bin_nights, y_true_nights, min_iou=0.3):
    total_tp = total_fp = total_fn = 0
    for yp, yt in zip(y_pred_bin_nights, y_true_nights):
        pred_events = extract_events_from_binary_mask(yp, fs=1)
        true_events = extract_events_from_binary_mask(yt, fs=1)
        tp, fp, fn = tp_fp_fn(pred_events, true_events, min_iou=min_iou)
        total_tp += tp; total_fp += fp; total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-12)
    recall    = total_tp / (total_tp + total_fn + 1e-12)
    return float(2 * precision * recall / (precision + recall + 1e-12))


In [ ]:
# y_valN deve ser (7, 18000) 0/1
y_true_val_nights = y_val.astype(int)

best_t, best_f1 = None, -1
for t in np.linspace(0.20, 0.80, 31):
    yb = (y_pred_val_nights >= t).astype(int)
    f1 = event_f1_nights(yb, y_true_val_nights, min_iou=0.3)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print("BEST threshold:", best_t)
print("BEST Event-F1:", best_f1)


In [ ]:
def postprocess_mask(mask, min_len=10, gap_fill=5):
    """
    mask: (T,) 0/1 em 1Hz
    min_len: remove eventos com duração < min_len segundos
    gap_fill: preenche buracos de zeros com duração <= gap_fill dentro de um evento
    """
    m = mask.astype(int).copy()
    T = len(m)

    # 1) fill small gaps (0-runs curtos entre 1s)
    i = 0
    while i < T:
        if m[i] == 0:
            j = i
            while j < T and m[j] == 0:
                j += 1
            gap = j - i
            left_one = (i - 1 >= 0 and m[i - 1] == 1)
            right_one = (j < T and m[j] == 1)
            if left_one and right_one and gap <= gap_fill:
                m[i:j] = 1
            i = j
        else:
            i += 1

    # 2) remove short events (1-runs curtos)
    i = 0
    while i < T:
        if m[i] == 1:
            j = i
            while j < T and m[j] == 1:
                j += 1
            run = j - i
            if run < min_len:
                m[i:j] = 0
            i = j
        else:
            i += 1

    return m


In [ ]:
best = (-1, None)

for t in np.linspace(0.2, 0.8, 31):
    raw = (y_pred_val_nights >= t).astype(int)

    for min_len in [1, 2, 3, 5, 8, 10, 12, 15]:
        for gap_fill in [0, 1, 2, 3, 5]:
            ypp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                            for i in range(raw.shape[0])])
            f1 = event_f1_nights(ypp, y_val.astype(int), min_iou=0.3)

            if f1 > best[0]:
                best = (f1, (t, min_len, gap_fill))

print("BEST postproc Event-F1:", best[0])
print("params (t, min_len, gap_fill):", best[1])
#print porcentage of 1 
t = best[1][0]
min_len = best[1][1]
gap_fill = best[1][2]

raw = (y_pred_val_nights >= t).astype(int)
y_pp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                 for i in range(raw.shape[0])])
print("Percentage of 1s after post-processing:", y_pp.mean() * 100)

In [ ]:

from itertools import product


In [ ]:
def moving_average(p, w):
    if w <= 1:
        return p
    kernel = np.ones(w) / w
    return np.convolve(p, kernel, mode="same")


In [ ]:
def postprocess_mask(mask, min_len=10, gap_fill=5):
    m = mask.astype(int).copy()
    T = len(m)

    # fill small gaps
    i = 0
    while i < T:
        if m[i] == 0:
            j = i
            while j < T and m[j] == 0:
                j += 1
            gap = j - i
            if i > 0 and j < T and m[i-1] == 1 and m[j] == 1 and gap <= gap_fill:
                m[i:j] = 1
            i = j
        else:
            i += 1

    # remove short events
    i = 0
    while i < T:
        if m[i] == 1:
            j = i
            while j < T and m[j] == 1:
                j += 1
            if j - i < min_len:
                m[i:j] = 0
            i = j
        else:
            i += 1

    return m


In [ ]:
def postprocess_from_probs(p, smooth_w, q, min_len, gap_fill):
    p = np.asarray(p, float)

    # 1) smoothing
    p_s = moving_average(p, smooth_w)

    # 2) threshold adaptativo por noite (percentil)
    t = np.percentile(p_s, q)
    raw = (p_s >= t).astype(int)

    # 3) gap fill + min_len
    return postprocess_mask(raw, min_len=min_len, gap_fill=gap_fill)


In [ ]:
def eval_on_val(y_pred_val_nights, y_true_val_nights, params):
    f1s = []
    for p_night, y_valN in zip(y_pred_val_nights, y_true_val_nights):
        ypp = postprocess_from_probs(p_night, **params)

        f1 = event_f1_nights(
            [ypp],                      # <<< aqui
            [y_valN.astype(int)],       # <<< aqui
            min_iou=0.3
        )
        f1s.append(f1)

    return float(np.mean(f1s))


In [ ]:
from itertools import product

results = []

grid = {
    "smooth_w": [1, 5, 10],          # segundos
    "q": [92, 94, 96, 97, 98, 99],   # percentil por noite
    "min_len": [3, 5, 8, 10, 12],    # segundos
    "gap_fill": [1, 3, 5],           # segundos
}

for smooth_w, q, min_len, gap_fill in product(
    grid["smooth_w"],
    grid["q"],
    grid["min_len"],
    grid["gap_fill"],
):
    params = dict(
        smooth_w=smooth_w,
        q=q,
        min_len=min_len,
        gap_fill=gap_fill,
    )

    f1 = eval_on_val(
        y_pred_val_nights,
        y_true_val_nights,
        params
    )

    results.append((f1, params))

results.sort(key=lambda x: x[0], reverse=True)

best_f1, best_params = results[0]
print("BEST VAL Event-F1:", best_f1)
print("BEST PARAMS:", best_params)
print("Percentage of 1s after post-processing:",
      np.mean([postprocess_from_probs(
          y_pred_val_nights[i], **best_params
      ).mean() for i in range(y_pred_val_nights.shape[0])]) * 100)


In [ ]:
import numpy as np
from itertools import product

def moving_average(p, w):
    if w <= 1:
        return p
    kernel = np.ones(w) / w
    return np.convolve(p, kernel, mode="same")

def postprocess_mask(mask, min_len=10, gap_fill=5):
    m = mask.astype(int).copy()
    T = len(m)

    # fill small gaps
    i = 0
    while i < T:
        if m[i] == 0:
            j = i
            while j < T and m[j] == 0:
                j += 1
            gap = j - i
            if i > 0 and j < T and m[i-1] == 1 and m[j] == 1 and gap <= gap_fill:
                m[i:j] = 1
            i = j
        else:
            i += 1

    # remove short events
    i = 0
    while i < T:
        if m[i] == 1:
            j = i
            while j < T and m[j] == 1:
                j += 1
            if j - i < min_len:
                m[i:j] = 0
            i = j
        else:
            i += 1

    return m

def postprocess_from_probs_z(p, smooth_w, z_thr, min_len, gap_fill):
    p = np.asarray(p, float)
    p_s = moving_average(p, smooth_w)

    mu = p_s.mean()
    sd = p_s.std() + 1e-8
    pz = (p_s - mu) / sd

    raw = (pz >= z_thr).astype(int)
    return postprocess_mask(raw, min_len=min_len, gap_fill=gap_fill)

def eval_on_val_z(y_pred_val_nights, y_true_val_nights, params):
    f1s = []
    for p_night, y_valN in zip(y_pred_val_nights, y_true_val_nights):
        ypp = postprocess_from_probs_z(p_night, **params)

        # sua função espera "lista de noites"
        f1 = event_f1_nights([ypp], [y_valN.astype(int)], min_iou=0.3)
        f1s.append(f1)
    return float(np.mean(f1s))

# ===== GRID =====
results = []

grid = {
    "smooth_w": [1, 3, 5, 10],             # s
    "z_thr":    [-0.5, 0.0, 0.5, 1.0, 1.5, 2.0],
    "min_len":  [2, 3, 5, 8, 10, 12],      # s
    "gap_fill": [1, 2, 3, 5],              # s
}

for smooth_w, z_thr, min_len, gap_fill in product(
    grid["smooth_w"], grid["z_thr"], grid["min_len"], grid["gap_fill"]
):
    params = dict(smooth_w=smooth_w, z_thr=z_thr, min_len=min_len, gap_fill=gap_fill)
    f1 = eval_on_val_z(y_pred_val_nights, y_true_val_nights, params)
    results.append((f1, params))

results.sort(key=lambda x: x[0], reverse=True)
best_f1, best_params = results[0]
print("BEST VAL Event-F1:", best_f1)
print("BEST PARAMS:", best_params)


In [ ]:
def tune_threshold_only(y_pred_val_nights, y_true_val_nights, min_len=12, gap_fill=2):
    best = (-1, None)
    for t in np.linspace(0.05, 0.95, 91):
        f1s = []
        for p, y in zip(y_pred_val_nights, y_true_val_nights):
            raw = (p >= t).astype(int)
            ypp = postprocess_mask(raw, min_len=min_len, gap_fill=gap_fill)
            f1 = event_f1_nights([ypp], [y.astype(int)], min_iou=0.3)
            f1s.append(f1)
        mean_f1 = float(np.mean(f1s))
        if mean_f1 > best[0]:
            best = (mean_f1, t)
    return best


In [ ]:
#qual a duracao minima e max de enventos em x_train
min_event_durations = []
for i in range(y.shape[0]):
    events = extract_events_from_binary_mask(y[i], fs=1)
    durations = [e - s for s, e in events]
    if durations:
        min_event_durations.append(min(durations)) 

max_event_durations = []
for i in range(y.shape[0]):
    events = extract_events_from_binary_mask(y[i], fs=1)
    durations = [e - s for s, e in events]
    if durations:
        max_event_durations.append(max(durations))  
print("Maximum event durations in training nights:", max_event_durations)    
print("Minimum event durations in training nights:", min_event_durations)
def apply_postproc_from_probs(y_pred_nights, t, min_len, gap_fill):
    raw = (y_pred_nights >= t).astype(int)
    y_pp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                     for i in range(raw.shape[0])])
    return y_pp


In [ ]:
# ✅ CÓDIGO CORRIGIDO
t, min_len, gap_fill = best[1]
raw = (y_pred_val_nights >= t).astype(int)
ypp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                for i in range(raw.shape[0])])
print(best_params)
# ✅ Comparação real vs predito
for i in range(ypp.shape[0]):  # Número de noites de validação
    n_true = len(extract_events_from_binary_mask(y_val[i]))
    n_pred = len(extract_events_from_binary_mask(ypp[i]))
    print(f"night {i}: "
          f"true_events={n_true:3d} | "
          f"pred_events={n_pred:3d} | "
          f"true%={y_val[i].mean()*100:5.2f}% | "
          f"pred%={ypp[i].mean()*100:5.2f}%")


In [ ]:
#quantos eventos foram perdifos em cada noite de validação com {'smooth_w': 3, 'z_thr': 0.5, 'min_len': 12, 'gap_fill': 3}
params = best_params
y_pp = []
for p_night in y_pred_val_nights:
    ypp = postprocess_from_probs_z(p_night, **params)
    y_pp.append(ypp)
    
y_pp = np.stack(y_pp)

for i in range(y_pp.shape[0]):
    pred_events = extract_events_from_binary_mask(y_pp[i], fs=1)
    true_events = extract_events_from_binary_mask(y_val[i], fs=1)
    tp, fp, fn = tp_fp_fn(pred_events, true_events, min_iou=0.3)
    print(f"night {i}: TP={tp:3d} | FP={fp:3d} | FN={fn:3d} | true%={y_val[i].mean()*100:5.2f}% | pred%={y_pp[i].mean()*100:5.2f}%")

In [ ]:
# ✅ CÓDIGO CORRIGIDO

#usar params (t, min_len, gap_fill): (0.54, 15, 5)


t = 0.54
min_len = 15
gap_fill = 5

raw = (y_pred_val_nights >= t).astype(int)
ypp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                for i in range(raw.shape[0])])
print("(t, min_len, gap_fill): (0.54, 15, 5)")
# ✅ Comparação real vs predito
for i in range(ypp.shape[0]):  # Número de noites de validação
    n_true = len(extract_events_from_binary_mask(y_val[i]))
    n_pred = len(extract_events_from_binary_mask(ypp[i]))
    print(f"night {i}: "
          f"true_events={n_true:3d} | "
          f"pred_events={n_pred:3d} | "
          f"true%={y_val[i].mean()*100:5.2f}% | "
          f"pred%={ypp[i].mean()*100:5.2f}%")

In [ ]:
#quantos eventos foram perdifos em cada noite de validação com (t, min_len, gap_fill): (0.54, 15, 5)
params = {
    "t": 0.52,
    "min_len": 8,
    "gap_fill": 5
}
y_pp = apply_postproc_from_probs(y_pred_val_nights, **params)
for i in range(y_pp.shape[0]):
    pred_events = extract_events_from_binary_mask(y_pp[i], fs=1)
    true_events = extract_events_from_binary_mask(y_val[i], fs=1)
    tp, fp, fn = tp_fp_fn(pred_events, true_events, min_iou=0.3)
    print(f"night {i}: TP={tp:3d} | FP={fp:3d} | FN={fn:3d} | true%={y_val[i].mean()*100:5.2f}% | pred%={y_pp[i].mean()*100:5.2f}%")

In [ ]:
best = (-1, None)

for t in np.linspace(0.10, 0.8, 16):
    raw = (y_pred_val_nights >= t).astype(int)

    for min_len in [3, 5, 8]:
        for gap_fill in [3, 5]:
            ypp = np.stack([
                postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                for i in range(raw.shape[0])
            ])

            f1 = event_f1_nights(ypp, y_val.astype(int), min_iou=0.3)

            if f1 > best[0]:
                best = (f1, (t, min_len, gap_fill))

print("BEST Event-F1:", best[0])
print("BEST params (t, min_len, gap_fill):", best[1])


In [ ]:
X_TEST_PATH = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_pre files\02_processed\nights_test_norm.h5"

with h5py.File(X_TEST_PATH, "r") as f:
    print(list(f.keys()))


In [ ]:

with h5py.File(X_TEST_PATH, "r") as f:
    Xn_test_nights = f["X_nights"][:].astype("float32")
    subj_order_test = f["subject_ids"][:].astype(int)

print("Xn_test_nights:", Xn_test_nights.shape)     # (22, 1800000, 8)
print("subj_order_test:", subj_order_test[:10], "...", subj_order_test[-10:])


In [ ]:
y_pred_test_nights = predict_nights_overlap_mean(
    model, Xn_test_nights, chunk_sec=300, stride_sec=60, fs=1, batch_size=4
)
print("y_pred_test_nights:", y_pred_test_nights.shape)  # (22, 18000)
print("min/max/mean:", y_pred_test_nights.min(), y_pred_test_nights.max(), y_pred_test_nights.mean())


In [ ]:
#y_pred_test_nights to csv file
y_pred_test_nights_df = pd.DataFrame(y_pred_test_nights)
y_pred_test_nights_df.insert(0, 'subject_id', subj_order_test)
y_pred_test_nights_df.to_csv("y_pred_test_nights.csv", index=False)
y_pred_test_nights_df.head(22)

In [ ]:
rows = []

for i, s in enumerate(subj_order_test):
    p = y_pred_test_nights[i]  # (18000,)
    for t, val in enumerate(p):
        rows.append({
            "subject": int(s),
            "t_sec": t,
            "p_apnea": float(val)
        })

night_df = pd.DataFrame(rows)

print(night_df.head())
print(night_df.describe())


In [ ]:
from collections import Counter

unique_h5 = np.unique(subj_test)
print("unique subjects in X_test.h5:", unique_h5)
print("n unique:", len(unique_h5))

counts = {int(s): int((subj_test==s).sum()) for s in unique_h5}
print("min/max windows per subject:", min(counts.values()), max(counts.values()))

# se subj_order_test existir:
print("subj_order_test:", subj_order_test, "len:", len(subj_order_test))

missing = [int(s) for s in subj_order_test if s not in set(unique_h5)]
extra   = [int(s) for s in unique_h5 if s not in set(subj_order_test)]
print("missing in X_test.h5 (present in subj_order_test):", missing)
print("extra in X_test.h5 (not in subj_order_test):", extra)


In [ ]:

# subjects reais do X_test.h5 (22..43)
subjects_sorted = np.sort(np.unique(subj_test))  # [22..43]
assert len(subjects_sorted) == 22
assert y_pred_test_nights.shape[0] == 22

# sua máscara pós-processada já criada:
# y_test_pp: (22,18000) e y_test_win: (22,200,90)
# se ainda não tiver, recria:
t = 0.535
min_len = 6
gap_fill = 3

raw = (y_pred_test_nights >= t).astype(int)
y_test_pp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                      for i in range(raw.shape[0])])
y_test_win = y_test_pp.reshape(22, 200, 90)

rows = []
for i, s in enumerate(subjects_sorted):
    idx = np.where(subj_test == s)[0]
    ids_s = ids_test[idx]                 # (200,)
    order = np.argsort(ids_s)
    ids_s = ids_s[order]

    y_s = y_test_win[i][order]            # (200,90) alinhado por ID

    df_s = pd.DataFrame(y_s, columns=[f"y_{k}" for k in range(90)])
    df_s.insert(0, "ID", ids_s)
    rows.append(df_s)

submission_df = pd.concat(rows, ignore_index=True).sort_values("ID").reset_index(drop=True)

mask_cols = [c for c in submission_df.columns if c.startswith("y_")]
print("submission shape:", submission_df.shape)         # (4400, 91)
print("IDs unique:", submission_df["ID"].is_unique)     # True
print("overall %ones:", submission_df[mask_cols].to_numpy().mean() * 100)

assert submission_df.shape[0] == len(ids_test), "Número de linhas não bate com X_test"
assert submission_df["ID"].is_unique, "IDs duplicados na submission"


In [ ]:
SUB_PATH = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\2_BaselineModel\3lcnn_night\pred_per_night_0.52_12_3.csv"
submission_df.to_csv(SUB_PATH, index=False)
print("✅ Saved:", SUB_PATH)


In [ ]:

min_len = 12
gap_fill = 3

def apply_postproc_from_probs(y_pred_nights, t, min_len=7, gap_fill=3):
    raw = (y_pred_nights >= t).astype(int)
    y_pp = np.stack([postprocess_mask(raw[i], min_len=min_len, gap_fill=gap_fill)
                     for i in range(raw.shape[0])])
    return y_pp

for t in [0.50, 0.51, 0.52, 0.53, 0.535, 0.54, 0.55, 0.57]:
    y_pp = apply_postproc_from_probs(y_pred_test_nights, t, min_len=min_len, gap_fill=gap_fill)
    print(f"t={t:.3f} -> %ones={y_pp.mean()*100:.3f}%")
